# SVM Model Selection

This notebook evaluates a linear Support Vector Machine classifier for the binary diabetes target using 5-fold stratified cross-validation. Metrics are computed directly from each validation fold. A linear SVM is used because the training dataset is large, and a full kernel SVM would be much slower.

In [1]:
import pandas as pd
from sklearn.base import clone
from sklearn.metrics import recall_score, precision_score, f1_score, fbeta_score, average_precision_score, roc_auc_score, roc_curve
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

In [2]:
# ==========================================
# 1. Load Training Data ONLY
# ==========================================
train_df = pd.read_csv('../data/processed/train.csv')

X_train = train_df.drop(columns=['Diabetes_01'])
y_train = train_df['Diabetes_01']

In [3]:
# ==========================================
# 2. Setup Stratified Cross-Validation
# ==========================================
cv_strategy = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [4]:
# ==========================================
# 3. Build Linear SVM Model
# ==========================================
svm_model = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("svm", LinearSVC(
        C=1.0,
        class_weight="balanced",
        max_iter=10000,
        random_state=42
    ))
])

In [5]:
# ==========================================
# 4. Run 5-Fold Cross-Validation Across Decision Thresholds
# ==========================================

# LinearSVC uses decision scores instead of probabilities.
# The default SVM classification threshold is 0. Lower thresholds increase recall.
thresholds = [0.0, -0.25, -0.5, -0.75, -1.0]

fold_metrics = []
fold_predictions = []

for fold_number, (train_idx, valid_idx) in enumerate(cv_strategy.split(X_train, y_train), start=1):
    estimator = clone(svm_model)

    X_fold_train = X_train.iloc[train_idx]
    y_fold_train = y_train.iloc[train_idx]
    X_valid = X_train.iloc[valid_idx]
    y_valid = y_train.iloc[valid_idx]

    estimator.fit(X_fold_train, y_fold_train)

    y_valid_score = estimator.decision_function(X_valid)

    for threshold in thresholds:
        y_valid_pred = (y_valid_score >= threshold).astype(int)

        fold_metrics.append({
            "Fold": fold_number,
            "Decision Threshold": threshold,
            "Validation Recall": recall_score(y_valid, y_valid_pred, pos_label=1, zero_division=0),
            "Validation Precision": precision_score(y_valid, y_valid_pred, pos_label=1, zero_division=0),
            "Validation F1": f1_score(y_valid, y_valid_pred, pos_label=1, zero_division=0),
            "Validation F2": fbeta_score(y_valid, y_valid_pred, beta=2, pos_label=1, zero_division=0),
            "Validation AUPRC": average_precision_score(y_valid, y_valid_score),
            "Validation AUROC": roc_auc_score(y_valid, y_valid_score),
            "Predicted Positive Rate": y_valid_pred.mean(),
        })

    fold_predictions.append(pd.DataFrame({
        "Fold": fold_number,
        "y_valid": y_valid.to_numpy(),
        "y_valid_score": y_valid_score,
    }))

svm_fold_metrics_df = pd.DataFrame(fold_metrics)
svm_cv_predictions_df = pd.concat(fold_predictions, ignore_index=True)

svm_fold_metrics_df.round(6)

,Fold,Decision Threshold,Validation Recall,Validation Precision,Validation F1,Validation F2,Validation AUPRC,Validation AUROC,Predicted Positive Rate
0,1,0.00,0.758504,0.315481,0.445619,0.592186,0.403412,0.806400,0.367197
1,1,-0.25,0.896527,0.259404,0.402382,0.601204,0.403412,0.806400,0.527839
2,1,-0.50,0.965628,0.214029,0.350394,0.567238,0.403412,0.806400,0.689052
3,1,-0.75,0.992342,0.183314,0.309461,0.527092,0.403412,0.806400,0.826765
4,1,-1.00,0.998931,0.164946,0.283140,0.496679,0.403412,0.806400,0.924929
5,2,0.00,0.763847,0.318837,0.449887,0.597154,0.405866,0.806892,0.365891
6,2,-0.25,0.894390,0.258813,0.401455,0.599799,0.405866,0.806892,0.527785
7,2,-0.50,0.960641,0.214823,0.351126,0.566966,0.405866,0.806892,0.682959
8,2,-0.75,0.989492,0.183016,0.308898,0.525957,0.405866,0.806892,0.825731
9,2,-1.00,0.997507,0.164653,0.282650,0.495866,0.405866,0.806892,0.925255


In [6]:
# ==========================================
# 5. Display Selected Cross-Validation Metrics
# ==========================================

svm_cv_summary_df = (
    svm_fold_metrics_df
    .groupby("Decision Threshold", as_index=False)
    .agg({
        "Validation Recall": "mean",
        "Validation Precision": "mean",
        "Validation F1": "mean",
        "Validation F2": "mean",
        "Validation AUPRC": "mean",
        "Validation AUROC": "mean",
        "Predicted Positive Rate": "mean",
    })
    .rename(columns={
        "Validation Recall": "Validation Recall Mean",
        "Validation Precision": "Validation Precision Mean",
        "Validation F1": "Validation F1 Mean",
        "Validation F2": "Validation F2 Mean",
        "Validation AUPRC": "Validation AUPRC Mean",
        "Validation AUROC": "Validation AUROC Mean",
        "Predicted Positive Rate": "Predicted Positive Rate Mean",
    })
)

default_threshold = 0.0
default_scores = svm_cv_summary_df.loc[
    svm_cv_summary_df["Decision Threshold"] == default_threshold
].copy()
default_scores.insert(0, "Selection Rule", "Default threshold")

best_f2_scores = svm_cv_summary_df.loc[
    [svm_cv_summary_df["Validation F2 Mean"].idxmax()]
].copy()
best_f2_scores.insert(0, "Selection Rule", "Max F2 threshold")

y_valid = svm_cv_predictions_df["y_valid"]
y_valid_score = svm_cv_predictions_df["y_valid_score"]
fpr, tpr, roc_thresholds = roc_curve(y_valid, y_valid_score)
best_tpr_fpr_index = (tpr - fpr).argmax()
best_tpr_fpr_threshold = roc_thresholds[best_tpr_fpr_index]
best_tpr_fpr_pred = (y_valid_score >= best_tpr_fpr_threshold).astype(int)

best_tpr_fpr_scores = pd.DataFrame([{
    "Selection Rule": "Max TPR-FPR threshold",
    "Decision Threshold": best_tpr_fpr_threshold,
    "Validation Recall Mean": recall_score(y_valid, best_tpr_fpr_pred, pos_label=1, zero_division=0),
    "Validation Precision Mean": precision_score(y_valid, best_tpr_fpr_pred, pos_label=1, zero_division=0),
    "Validation F1 Mean": f1_score(y_valid, best_tpr_fpr_pred, pos_label=1, zero_division=0),
    "Validation F2 Mean": fbeta_score(y_valid, best_tpr_fpr_pred, beta=2, pos_label=1, zero_division=0),
    "Validation AUPRC Mean": average_precision_score(y_valid, y_valid_score),
    "Validation AUROC Mean": roc_auc_score(y_valid, y_valid_score),
    "Predicted Positive Rate Mean": best_tpr_fpr_pred.mean(),
    "TPR - FPR": tpr[best_tpr_fpr_index] - fpr[best_tpr_fpr_index],
}])

svm_selected_metrics_df = pd.concat(
    [default_scores, best_f2_scores, best_tpr_fpr_scores],
    ignore_index=True
)

svm_selected_metrics_df.round(6)


,Selection Rule,Decision Threshold,Validation Recall Mean,Validation Precision Mean,Validation F1 Mean,Validation F2 Mean,Validation AUPRC Mean,Validation AUROC Mean,Predicted Positive Rate Mean,TPR - FPR
0,Default threshold,0.000000,0.758949,0.316465,0.446673,0.593090,0.404646,0.806097,0.366307,NaN
1,Max F2 threshold,-0.250000,0.893472,0.258814,0.401363,0.599469,0.404646,0.806097,0.527282,NaN
2,Max TPR-FPR threshold,-0.029723,0.779464,0.309237,0.442801,0.597693,0.404312,0.806094,0.384993,0.465583
